In [16]:
import pandas as pd
import sqlite3
df = pd.read_csv(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.csv', na_values=['NULL', 'null', 'NA', ''])

In [17]:
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

# Table 1 bookings:
bookings = df[[
    'hotel', 'is_canceled', 'lead_time',
    'arrival_date_year', 'arrival_date_month',
    'arrival_date_week_number', 'arrival_date_day_of_month',
    'stays_in_weekend_nights', 'stays_in_week_nights',
    'adr', 'booking_changes', 'days_in_waiting_list',
    'required_car_parking_spaces', 'total_of_special_requests',
    'reservation_status', 'reservation_status_date'
]].copy()
bookings.index.name = 'booking_id'
bookings.reset_index(inplace=True)
bookings.to_sql('bookings', conn, if_exists='replace', index=False)

# Table 2 customers:
customers = df[[
    'country', 'is_repeated_guest',
    'previous_cancellations',
    'previous_bookings_not_canceled',
    'adults', 'children', 'babies'
]].copy()
customers.index.name = 'booking_id'
customers.reset_index(inplace=True)
customers.to_sql('customers', conn, if_exists='replace', index=False)

# Table 3 reservations:
reservations = df[[
    'meal', 'market_segment', 'distribution_channel',
    'reserved_room_type', 'assigned_room_type',
    'deposit_type', 'agent', 'company', 'customer_type'
]].copy()
reservations.index.name = 'booking_id'
reservations.reset_index(inplace=True)
reservations.to_sql('reservations', conn, if_exists='replace', index=False)

conn.close()

In [18]:
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

# Q1: Which hotel type has the higher cancellation rate?
q1 = pd.read_sql_query("""
    SELECT 
        hotel,
        COUNT(*) AS total_bookings,
        SUM(is_canceled) AS total_cancellations,
        ROUND(SUM(is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct
    FROM bookings
    GROUP BY hotel
    ORDER BY cancellation_rate_pct DESC
""", conn)

print("Q1 — Cancellation Rate by Hotel Type")
print("=" * 45)
print(q1.to_string(index=False))

Q1 — Cancellation Rate by Hotel Type
       hotel  total_bookings  total_cancellations  cancellation_rate_pct
  City Hotel           79330                33102                  41.73
Resort Hotel           40060                11122                  27.76


Question1:
### Interpretation
City Hotel cancels at 41.73% vs Resort Hotel at 27.76% , a 14 point gap driven by different booking profiles. City hotels attract volatile business and last-minute bookings while resort bookings represent committed leisure trips. City Hotel urgently needs stricter deposit policies to protect revenue.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [20]:
# Q2: Average daily rate by hotel type and year
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q2 = pd.read_sql_query("""
    SELECT 
        hotel,
        arrival_date_year,
        COUNT(*) AS total_bookings,
        ROUND(AVG(adr), 2) AS avg_daily_rate
    FROM bookings
    GROUP BY hotel, arrival_date_year
    ORDER BY hotel, arrival_date_year
""", conn)

print("Q2 — Average Daily Rate by Hotel Type and Year")
print("=" * 50)
print(q2.to_string(index=False))

Q2 — Average Daily Rate by Hotel Type and Year
       hotel  arrival_date_year  total_bookings  avg_daily_rate
  City Hotel               2015           13682           85.86
  City Hotel               2016           38140          103.48
  City Hotel               2017           27508          117.50
Resort Hotel               2015            8314           89.35
Resort Hotel               2016           18567           87.73
Resort Hotel               2017           13179          108.66


Question2:
### Interpretation
City Hotel ADR grew 37% from 2015 to 2017 ($85.86 → $117.50) showing consistent pricing power. Resort Hotel was flat from 2015 to 2016 then spiked in 2017. By 2017 City Hotel commands a $9 premium over Resort Hotel, unusual since resorts typically charge more, suggesting stronger urban demand growth.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [21]:
# Q3: Which market segment loses the most revenue due to cancellations?
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q3 = pd.read_sql_query("""
    SELECT 
        r.market_segment,
        COUNT(*) AS total_bookings,
        SUM(b.is_canceled) AS total_cancellations,
        ROUND(SUM(b.is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
        ROUND(AVG(b.adr), 2) AS avg_daily_rate,
        ROUND(SUM(CASE WHEN b.is_canceled = 1 
            THEN b.adr * (b.stays_in_weekend_nights + b.stays_in_week_nights) 
            ELSE 0 END), 2) AS lost_revenue
    FROM bookings b
    JOIN reservations r ON b.booking_id = r.booking_id
    GROUP BY r.market_segment
    ORDER BY lost_revenue DESC
""", conn)

print("Q3 — Revenue Lost to Cancellations by Market Segment")
print("=" * 55)
print(q3.to_string(index=False))

Q3 — Revenue Lost to Cancellations by Market Segment
market_segment  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate  lost_revenue
     Online TA           56477                20739                  36.72          117.20   10227646.11
        Groups           19811                12097                  61.06           79.48    2800543.98
 Offline TA/TO           24219                 8311                  34.32           87.35    2492358.15
        Direct           12606                 1934                  15.34          115.45     993409.82
     Corporate            5295                  992                  18.73           69.36     196383.07
      Aviation             237                   52                  21.94          100.14      16578.00
 Complementary             743                   97                  13.06            2.89        269.99
     Undefined               2                    2                 100.00           15.00         48.00


Question3:
### Interpretation
Online TA loses the most revenue in absolute terms ($10.2M) due to its large volume (56,477 bookings) despite a moderate 37% cancellation rate. Groups is the most dangerous segment with a 61% cancellation rate, 6 in 10 group bookings cancel. Direct bookings are the most reliable with only 15.34% cancellation rate and the second highest ADR at $115.45.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [22]:
# Q4: Top 10 countries by bookings with cancellation rate and ADR
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q4 = pd.read_sql_query("""
    SELECT 
        c.country,
        COUNT(*) AS total_bookings,
        SUM(b.is_canceled) AS total_cancellations,
        ROUND(SUM(b.is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
        ROUND(AVG(b.adr), 2) AS avg_daily_rate
    FROM bookings b
    JOIN customers c ON b.booking_id = c.booking_id
    WHERE c.country IS NOT NULL
    GROUP BY c.country
    ORDER BY total_bookings DESC
    LIMIT 10
""", conn)

print("Q4 — Top 10 Countries by Bookings")
print("=" * 55)
print(q4.to_string(index=False))

Q4 — Top 10 Countries by Bookings
country  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate
    PRT           48590                27519                  56.64           92.04
    GBR           12129                 2453                  20.22           96.02
    FRA           10415                 1934                  18.57          109.62
    ESP            8568                 2177                  25.41          117.00
    DEU            7287                 1218                  16.71          104.40
    ITA            3766                 1333                  35.40          113.95
    IRL            3375                  832                  24.65           98.19
    BEL            2342                  474                  20.24          113.85
    BRA            2224                  830                  37.32          111.01
    NLD            2104                  387                  18.39          108.09


Question4:
### Interpretation
Portugal (domestic market) dominates with 48,590 bookings but has a shocking 56.64% cancellation rate, the highest of any top market. UK and French guests are far more reliable at 20% and 18% respectively. Stricter deposit requirements should be applied specifically to domestic bookings which represent both the largest volume and highest cancellation risk.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [28]:
# Q5: Which deposit type has the highest cancellation rate?
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q5 = pd.read_sql_query("""
    SELECT 
        r.deposit_type,
        COUNT(*) AS total_bookings,
        SUM(b.is_canceled) AS total_cancellations,
        ROUND(SUM(b.is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct
    FROM bookings b
    JOIN reservations r ON b.booking_id = r.booking_id
    GROUP BY r.deposit_type
    ORDER BY cancellation_rate_pct DESC
""", conn)

print("Q5 — Cancellation Rate by Deposit Type")
print("=" * 45)
print(q5.to_string(index=False))

Q5 — Cancellation Rate by Deposit Type
deposit_type  total_bookings  total_cancellations  cancellation_rate_pct
  Non Refund           14587                14494                  99.36
  No Deposit          104641                29694                  28.38
  Refundable             162                   36                  22.22


Question5:
### Interpretation
The 99.36% cancellation rate for Non Refund deposits is counterintuitive but explainable, hotels record these as cancelled even though the deposit is retained, meaning revenue is protected regardless of the cancellation flag. No Deposit bookings at 28.38% represent the true behavioural cancellation risk. Expanding non-refundable rate options for high-risk segments would protect revenue without reducing bookings.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [30]:
# Q6: Monthly booking trends across 2015-2017
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q6 = pd.read_sql_query("""
    SELECT 
        arrival_date_year,
        arrival_date_month,
        CASE arrival_date_month
            WHEN 'January' THEN 1 WHEN 'February' THEN 2
            WHEN 'March' THEN 3 WHEN 'April' THEN 4
            WHEN 'May' THEN 5 WHEN 'June' THEN 6
            WHEN 'July' THEN 7 WHEN 'August' THEN 8
            WHEN 'September' THEN 9 WHEN 'October' THEN 10
            WHEN 'November' THEN 11 WHEN 'December' THEN 12
        END AS month_num,
        COUNT(*) AS total_bookings,
        SUM(is_canceled) AS total_cancellations,
        ROUND(SUM(is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
        ROUND(AVG(adr), 2) AS avg_daily_rate,
        SUM(COUNT(*)) OVER (
            PARTITION BY arrival_date_year
            ORDER BY CASE arrival_date_month
                WHEN 'January' THEN 1 WHEN 'February' THEN 2
                WHEN 'March' THEN 3 WHEN 'April' THEN 4
                WHEN 'May' THEN 5 WHEN 'June' THEN 6
                WHEN 'July' THEN 7 WHEN 'August' THEN 8
                WHEN 'September' THEN 9 WHEN 'October' THEN 10
                WHEN 'November' THEN 11 WHEN 'December' THEN 12
            END
        ) AS running_total_bookings
    FROM bookings
    GROUP BY arrival_date_year, arrival_date_month
    ORDER BY arrival_date_year, month_num
""", conn)

print("Q6 — Monthly Booking Trends 2015-2017")
print("=" * 70)
print(q6.to_string(index=False))

Q6 — Monthly Booking Trends 2015-2017
 arrival_date_year arrival_date_month  month_num  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate  running_total_bookings
              2015               July          7            2776                 1259                  45.35           97.83                    2776
              2015             August          8            3889                 1598                  41.09          105.92                    6665
              2015          September          9            5114                 2094                  40.95           94.82                   11779
              2015            October         10            4957                 1732                  34.94           78.90                   16736
              2015           November         11            2340                  486                  20.77           60.58                   19076
              2015           December         12            2920    

Question6:
### Interpretation
Summer months (July–August) consistently show peak ADR reaching $164.25 in August 2017. Cancellation rates are higher in peak months (40–45%) than winter months (20–35%), suggesting peak season attracts more speculative bookings. Strong year-over-year growth in both bookings and ADR confirms healthy demand expansion across the 2015–2017 period.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [29]:
# Q7: Does lead time affect cancellation rate?
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q7 = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN lead_time = 0 THEN '1. Same day'
            WHEN lead_time <= 7 THEN '2. 1-7 days'
            WHEN lead_time <= 30 THEN '3. 8-30 days'
            WHEN lead_time <= 90 THEN '4. 31-90 days'
            WHEN lead_time <= 180 THEN '5. 91-180 days'
            ELSE '6. 180+ days'
        END AS lead_time_bucket,
        COUNT(*) AS total_bookings,
        SUM(is_canceled) AS total_cancellations,
        ROUND(SUM(is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
        ROUND(AVG(adr), 2) AS avg_daily_rate
    FROM bookings
    GROUP BY lead_time_bucket
    ORDER BY cancellation_rate_pct DESC
""", conn)

print("Q7 — Cancellation Rate by Lead Time Bucket")
print("=" * 55)
print(q7.to_string(index=False))

Q7 — Cancellation Rate by Lead Time Bucket
lead_time_bucket  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate
    6. 180+ days           24692                14077                  57.01           93.00
  5. 91-180 days           26439                11821                  44.71          109.13
   4. 31-90 days           29553                11141                  37.70          106.57
    3. 8-30 days           18960                 5283                  27.86          108.12
     2. 1-7 days           13401                 1472                  10.98           93.15
     1. Same day            6345                  430                   6.78           83.25


Question7:
### Interpretation
A perfectly linear relationship exists between lead time and cancellation rate, bookings made 180+ days in advance cancel at 57% while same-day bookings cancel at only 6.78%. Longer time horizons give guests more opportunity for plans to change. This directly supports a lead-time based deposit policy requiring non-refundable terms for all bookings made more than 90 days in advance.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [25]:
# Q8: Do repeat guests behave differently from new guests?
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q8 = pd.read_sql_query("""
    WITH guest_stats AS (
        SELECT 
            c.is_repeated_guest,
            COUNT(*) AS total_bookings,
            SUM(b.is_canceled) AS total_cancellations,
            ROUND(SUM(b.is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
            ROUND(AVG(b.adr), 2) AS avg_daily_rate,
            ROUND(AVG(b.total_of_special_requests), 2) AS avg_special_requests
        FROM bookings b
        JOIN customers c ON b.booking_id = c.booking_id
        GROUP BY c.is_repeated_guest
    )
    SELECT 
        CASE WHEN is_repeated_guest = 1 
             THEN 'Repeat Guest' 
             ELSE 'New Guest' 
        END AS guest_type,
        total_bookings,
        total_cancellations,
        cancellation_rate_pct,
        avg_daily_rate,
        avg_special_requests
    FROM guest_stats
    ORDER BY is_repeated_guest DESC
""", conn)

print("Q8 — Repeat vs New Guest Behaviour")
print("=" * 55)
print(q8.to_string(index=False))

Q8 — Repeat vs New Guest Behaviour
  guest_type  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate  avg_special_requests
Repeat Guest            3810                  552                  14.49           64.45                  0.63
   New Guest          115580                43672                  37.79          103.06                  0.57


Question8:
### Interpretation
Guests who received a different room than reserved cancel at only 5.38% vs 41.56% for matched rooms, a counterintuitive finding explained by timing. Room changes are discovered at check-in after arrival when cancellation is no longer practical. The lower ADR for changed rooms ($83.36 vs $104.47) suggests most changes are downgrades from premium to standard rooms.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [26]:
# Q9: Does getting a different room than reserved affect cancellation?
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q9 = pd.read_sql_query("""
    SELECT 
        CASE WHEN r.reserved_room_type = r.assigned_room_type 
             THEN 'Room Matched' 
             ELSE 'Room Changed' 
        END AS room_match_status,
        COUNT(*) AS total_bookings,
        SUM(b.is_canceled) AS total_cancellations,
        ROUND(SUM(b.is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
        ROUND(AVG(b.adr), 2) AS avg_daily_rate
    FROM reservations r
    JOIN bookings b ON r.booking_id = b.booking_id
    GROUP BY room_match_status
    ORDER BY cancellation_rate_pct DESC
""", conn)

print("Q9 — Room Match vs Room Change: Cancellation Impact")
print("=" * 55)
print(q9.to_string(index=False))

Q9 — Room Match vs Room Change: Cancellation Impact
room_match_status  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate
     Room Matched          104473                43422                  41.56          104.47
     Room Changed           14917                  802                   5.38           83.36


Question9:
### Interpretation
Guests who received a different room than reserved cancel at only 5.38% vs 41.56% for matched rooms, a counterintuitive finding explained by timing. Room changes are discovered at check-in after arrival when cancellation is no longer practical. The lower ADR for changed rooms ($83.36 vs $104.47) suggests most changes are downgrades from premium to standard rooms.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [27]:
# Q10: Do high-value bookings cancel more or less than low-value ones?
conn = sqlite3.connect(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\data\hotel_bookings.db')

q10 = pd.read_sql_query("""
    WITH booking_revenue AS (
        SELECT 
            booking_id,
            hotel,
            is_canceled,
            adr,
            (stays_in_weekend_nights + stays_in_week_nights) AS total_nights,
            ROUND(adr * (stays_in_weekend_nights + stays_in_week_nights), 2) AS total_revenue,
            NTILE(4) OVER (
                ORDER BY adr * (stays_in_weekend_nights + stays_in_week_nights)
            ) AS revenue_quartile
        FROM bookings
        WHERE adr > 0
    )
    SELECT 
        CASE revenue_quartile
            WHEN 1 THEN 'Q1 - Low Value'
            WHEN 2 THEN 'Q2 - Mid-Low Value'
            WHEN 3 THEN 'Q3 - Mid-High Value'
            WHEN 4 THEN 'Q4 - High Value'
        END AS booking_tier,
        COUNT(*) AS total_bookings,
        SUM(is_canceled) AS total_cancellations,
        ROUND(SUM(is_canceled) * 100.0 / COUNT(*), 2) AS cancellation_rate_pct,
        ROUND(AVG(adr), 2) AS avg_daily_rate,
        ROUND(AVG(total_nights), 1) AS avg_nights,
        ROUND(AVG(total_revenue), 2) AS avg_revenue
    FROM booking_revenue
    GROUP BY revenue_quartile
    ORDER BY revenue_quartile
""", conn)

print("Q10 — Cancellation Rate by Booking Value Quartile")
print("=" * 60)
print(q10.to_string(index=False))

Q10 — Cancellation Rate by Booking Value Quartile
       booking_tier  total_bookings  total_cancellations  cancellation_rate_pct  avg_daily_rate  avg_nights  avg_revenue
     Q1 - Low Value           29358                10188                  34.70           72.94         1.5        97.52
 Q2 - Mid-Low Value           29358                11263                  38.36           93.51         2.6       212.20
Q3 - Mid-High Value           29357                11290                  38.46          105.61         3.7       348.65
    Q4 - High Value           29357                11270                  38.39          142.07         6.1       796.93


Question10:
### Interpretation
Cancellation rates are remarkably uniform across all four value quartiles (34.70%–38.46%) booking value does not predict cancellation likelihood. However when a Q4 high-value booking cancels the average revenue loss is $796.93 vs $97.52 for Q1 an 8x difference. This means the strictest deposit policies should target high-value bookings not because they cancel more often but because the financial impact when they do is dramatically higher.

------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [32]:
import os
os.makedirs(r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\sql\results', exist_ok=True)

results = {
    'Q1_cancellation_by_hotel': q1,
    'Q2_adr_by_hotel_year': q2,
    'Q3_revenue_lost_by_segment': q3,
    'Q4_top_countries': q4,
    'Q5_cancellation_by_deposit': q5,
    'Q6_monthly_trends': q6,
    'Q7_lead_time_buckets': q7,
    'Q8_repeat_vs_new_guest': q8,
    'Q9_room_match_impact': q9,
    'Q10_booking_value_quartiles': q10
}

base_path = r'C:\Users\sadeq\Desktop\Python\Hotel_Booking\sql\results'
for name, df in results.items():
    df.to_csv(f'{base_path}\\{name}.csv', index=False)
    print(f"Saved: {name}.csv")

conn.close()

Saved: Q1_cancellation_by_hotel.csv
Saved: Q2_adr_by_hotel_year.csv
Saved: Q3_revenue_lost_by_segment.csv
Saved: Q4_top_countries.csv
Saved: Q5_cancellation_by_deposit.csv
Saved: Q6_monthly_trends.csv
Saved: Q7_lead_time_buckets.csv
Saved: Q8_repeat_vs_new_guest.csv
Saved: Q9_room_match_impact.csv
Saved: Q10_booking_value_quartiles.csv
